# S10 — Jhonatan | Predicción de velocidad de la próxima observación
**Proyecto Sello, Unidad II · CRISP-DM e inferencia sin estado.**

**Pregunta:** ¿Cuál será la velocidad de una aeronave en su siguiente observación de OpenSky, usando únicamente la lectura actual?

**Alcance:** siguientes observaciones del mismo ICAO24 registradas con separación temporal controlada (máximo 30 min). No equivale a velocidad dentro de exactamente cinco minutos, porque el muestreo OpenSky no garantiza esa frecuencia. La inferencia en vivo sí es sin estado, análoga a la guía S10.

## Fase 1. Business Understanding

In [1]:
NOMBRE_APP = 'S10-Jhonatan-Velocidad'
from pathlib import Path
import json, math
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, DoubleType)
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

ROOT = '/workspace'
RUTA = f'{ROOT}/artifacts/s8/parquet_aeronaves_final'
PAQUETE = 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0'
spark = (SparkSession.builder.appName(NOMBRE_APP).master('local[2]')
    .config('spark.jars.packages', PAQUETE)
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.sql.shuffle.partitions', '3').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version, '| Data Lake:', RUTA)


:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-220dcaa5-c709-4475-9088-37246e29129c;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark: 4.2.0 | Data Lake: /workspace/artifacts/s8/parquet_aeronaves_final


## Fase 2. Data Understanding

In [2]:
df=spark.read.parquet(RUTA)
df.printSchema()
print('Observaciones:',df.count(),'ICAO24:',df.select('icao24').distinct().count())
df.select(F.min('eventTimestamp'),F.max('eventTimestamp')).show()
df.groupBy('icao24').count().orderBy(F.desc('count')).show(20)


root
 |-- eventId: string (nullable = true)
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- timePosition: long (nullable = true)
 |-- eventTimestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- baroAltitude: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- verticalRate: double (nullable = true)
 |-- source: string (nullable = true)
 |-- kafkaPartition: integer (nullable = true)
 |-- kafkaOffset: long (nullable = true)



Observaciones: 248 ICAO24: 49


+-------------------+-------------------+
|min(eventTimestamp)|max(eventTimestamp)|
+-------------------+-------------------+
|2026-10-30 15:16:47|2026-10-30 19:58:07|
+-------------------+-------------------+



[Stage 13:======================================>                   (2 + 1) / 3]

+------+-----+
|icao24|count|
+------+-----+
|e80483|   12|
|e80481|   12|
|e8047a|   11|
|e80326|   11|
|e8026f|   10|
|e8046a|   10|
|e80272|   10|
|e80482|    9|
|e8027d|    8|
|e80412|    8|
|e80480|    8|
|a18b9d|    7|
|0ac333|    7|
|e8021b|    7|
|e8060b|    7|
|e49c05|    7|
|e80484|    6|
|e8033a|    6|
|e8033d|    5|
|e80479|    5|
+------+-----+
only showing top 20 rows


## Fase 3. Data Preparation — `lead(velocity,1)` particionado por ICAO24
Se predice la velocidad de la próxima posición de **la misma aeronave**, evitando emparejar aeronaves diferentes. Se exige un intervalo de observación positivo y menor o igual a 30 minutos. Los ejemplos con valores nulos o inválidos se excluyen.

In [3]:
base_df=(df.filter(F.col('icao24').rlike('^[0-9a-f]{6}$'))
   .filter(F.col('eventTimestamp').isNotNull())
   .filter(F.col('velocity').between(0,400))
   .filter(F.col('latitude').between(-90,90)&F.col('longitude').between(-180,180))
   .fillna({'baroAltitude':0.0,'verticalRate':0.0})
   .dropDuplicates(['icao24','timePosition']))
w = Window.partitionBy('icao24').orderBy('eventTimestamp','timePosition')
dataset=(base_df.withColumn('siguiente_fecha',F.lead('eventTimestamp',1).over(w))
    .withColumn('label',F.lead('velocity',1).over(w).cast('double'))
    .withColumn('separacion_seg',F.unix_timestamp('siguiente_fecha')-F.unix_timestamp('eventTimestamp'))
    .filter(F.col('separacion_seg').between(1,1800))
    .withColumn('hora',F.hour('eventTimestamp').cast('double'))
    .withColumn('dia_semana',F.dayofweek('eventTimestamp').cast('double'))
    .withColumn('hora_sin',F.sin(F.col('hora')*F.lit(2*math.pi/24)))
    .withColumn('hora_cos',F.cos(F.col('hora')*F.lit(2*math.pi/24)))
    .withColumn('persistencia',F.col('velocity').cast('double'))
    .orderBy('eventTimestamp'))
FEATURES=['velocity','baroAltitude','verticalRate','latitude','longitude','hora_sin','hora_cos','dia_semana']
filas=dataset.count()
print('Pares de observaciones válidos:',filas)
dataset.select('icao24','eventTimestamp','siguiente_fecha','velocity','label','separacion_seg').show(20,False)
if filas < 30:
    raise RuntimeError(f'Solo {filas} pares válidos: es insuficiente para una evaluación defendible. Recopila más observaciones por aeronave.')


Pares de observaciones válidos: 189


[Stage 26:======================================>                   (2 + 1) / 3]

+------+-------------------+-------------------+--------+------+--------------+
|icao24|eventTimestamp     |siguiente_fecha    |velocity|label |separacion_seg|
+------+-------------------+-------------------+--------+------+--------------+
|a20307|2026-10-30 15:16:47|2026-10-30 15:20:13|183.66  |204.88|206           |
|a18b9d|2026-10-30 15:17:47|2026-10-30 15:19:55|111.64  |119.36|128           |
|e80261|2026-10-30 15:17:49|2026-10-30 15:19:55|184.86  |177.72|126           |
|e80270|2026-10-30 15:17:49|2026-10-30 15:19:55|217.97  |216.85|126           |
|ad14aa|2026-10-30 15:17:50|2026-10-30 15:19:55|243.86  |227.01|125           |
|e8026f|2026-10-30 15:17:50|2026-10-30 15:19:55|115.23  |80.53 |125           |
|e8032f|2026-10-30 15:17:50|2026-10-30 15:19:55|215.53  |225.94|125           |
|e80483|2026-10-30 15:17:50|2026-10-30 15:19:55|104.91  |145.86|125           |
|e80413|2026-10-30 15:19:43|2026-10-30 15:21:48|220.69  |200.21|125           |
|a18b9d|2026-10-30 15:19:55|2026-10-30 1

### División cronológica 80/20 y prevención de fuga del objetivo

In [4]:
# Asignar train/test por fecha de la *etiqueta*, no solo por fecha de origen.
orden=Window.orderBy('siguiente_fecha','icao24','eventTimestamp')
partes=dataset.withColumn('_orden',F.row_number().over(orden))
limite=int(filas*.8)
train=partes.filter(F.col('_orden')<=limite).drop('_orden').cache()
test=partes.filter(F.col('_orden')>limite).drop('_orden').cache()
print('Train:',train.count(),'Test:',test.count())
train.agg(F.max('siguiente_fecha')).show(); test.agg(F.min('siguiente_fecha')).show()


Train: 151 Test: 38
+--------------------+
|max(siguiente_fecha)|
+--------------------+
| 2026-10-30 19:50:00|
+--------------------+

+--------------------+
|min(siguiente_fecha)|
+--------------------+
| 2026-10-30 19:50:02|
+--------------------+



## Fases 4 y 5. Baseline y modelos MLlib

In [5]:
# 4. Modelado y 5. Evaluación. El test es estrictamente posterior al train.
def evaluar(df, columna='prediction'):
    return {k: RegressionEvaluator(labelCol='label', predictionCol=columna, metricName=k).evaluate(df)
            for k in ('rmse', 'mae', 'r2')}

base = evaluar(test.withColumn('prediction', F.col('persistencia')))
print('Baseline persistencia:', base)
modelos = {}
for nombre, estimador in [
    ('LinearRegression', LinearRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.01)),
    ('RandomForest', RandomForestRegressor(featuresCol='features', labelCol='label', numTrees=40, maxDepth=5, seed=42))
]:
    pipeline = Pipeline(stages=[VectorAssembler(inputCols=FEATURES, outputCol='features', handleInvalid='skip'), estimador])
    modelo = pipeline.fit(train)
    met = evaluar(modelo.transform(test))
    modelos[nombre] = (modelo, met)
    print(nombre, met)

ordenados = sorted(modelos, key=lambda k: modelos[k][1]['rmse'])
mejor_ml = ordenados[0]
seleccion = 'Persistencia' if base['rmse'] <= modelos[mejor_ml][1]['rmse'] else mejor_ml
print('Mejor candidato segun RMSE:', seleccion)
print('ATENCIÓN: evaluar R2, MAE, estabilidad y tamaño del test antes de confiar en el modelo.')


Baseline persistencia: {'rmse': 21.95003788511297, 'mae': 13.591052631578949, 'r2': 0.8334448202220303}


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


LinearRegression {'rmse': 19.02938260877682, 'mae': 12.198060606732929, 'r2': 0.8748194107040961}
RandomForest {'rmse': 19.4220357084347, 'mae': 11.656486530869644, 'r2': 0.869600150016154}
Mejor candidato segun RMSE: LinearRegression
ATENCIÓN: evaluar R2, MAE, estabilidad y tamaño del test antes de confiar en el modelo.


## Fase 6. Guardar y recargar PipelineModel

In [6]:
MODEL_DIR=f'{ROOT}/artifacts/s10/models/jhonatan_velocidad_ml'
modelos[mejor_ml][0].write().overwrite().save(MODEL_DIR)
recargado=PipelineModel.load(MODEL_DIR)
recargado.transform(test).select('icao24','eventTimestamp','label','prediction').show(10,False)
Path(f'{ROOT}/artifacts/s10').mkdir(parents=True,exist_ok=True)
Path(f'{ROOT}/artifacts/s10/seleccion_jhonatan.json').write_text(json.dumps({
    'elegido':seleccion,'mejor_ml':mejor_ml,'baseline':base,
    'modelos':{k:v[1] for k,v in modelos.items()},'pares_validos':filas
},indent=2))
print('Modelo guardado:',MODEL_DIR)


+------+-------------------+------+------------------+
|icao24|eventTimestamp     |label |prediction        |
+------+-------------------+------+------------------+
|e80326|2026-10-30 19:48:05|227.1 |231.40363552481915|
|e80480|2026-10-30 19:44:36|228.14|243.3041389188061 |
|e8047a|2026-10-30 19:48:05|204.78|211.34535425995662|
|e80480|2026-10-30 19:50:02|228.64|240.75014780008013|
|e80481|2026-10-30 19:49:37|245.23|245.22728204415546|
|e80414|2026-10-30 19:50:02|140.66|108.33436598738297|
|e8024f|2026-10-30 19:49:50|65.12 |76.63358606210292 |
|e8026f|2026-10-30 19:49:25|88.58 |120.54276309081268|
|e80326|2026-10-30 19:50:02|229.84|233.09267972433784|
|e8033a|2026-10-30 19:49:59|169.12|167.7548341552401 |
+------+-------------------+------+------------------+
only showing top 10 rows
Modelo guardado: /workspace/artifacts/s10/models/jhonatan_velocidad_ml


## Fase 7. Inferencia sin estado desde Kafka
Cada evento contiene los predictores necesarios para estimar la velocidad de la siguiente observación (su llegada puede ser irregular). La predicción se guarda en Parquet. La fuente es `aeronaves-telemetria` con el contrato v1.

In [7]:
ESQUEMA = StructType([
    StructField('eventId',StringType()),StructField('eventType',StringType()),
    StructField('schemaVersion',IntegerType()),StructField('icao24',StringType()),
    StructField('eventTime',StringType()),StructField('latitude',DoubleType()),
    StructField('longitude',DoubleType()),StructField('baroAltitude',DoubleType()),
    StructField('velocity',DoubleType()),StructField('verticalRate',DoubleType()),
    StructField('source',StringType())])
raw=(spark.readStream.format('kafka').option('kafka.bootstrap.servers','kafka:9092')
    .option('subscribe','aeronaves-telemetria').option('startingOffsets','latest').load())
vivos=(raw.select(F.from_json(F.col('value').cast('string'),ESQUEMA).alias('d')).select('d.*')
    .withColumn('eventTimestamp',F.to_timestamp('eventTime'))
    .filter((F.col('eventType')=='aeronave.telemetria')&(F.col('schemaVersion')==1)&(F.col('source')=='opensky'))
    .filter(F.col('eventTimestamp').isNotNull()&F.col('icao24').rlike('^[0-9a-f]{6}$'))
    .filter(F.col('velocity').between(0,400)&F.col('latitude').between(-90,90)&F.col('longitude').between(-180,180))
    .fillna({'baroAltitude':0.0,'verticalRate':0.0})
    .withColumn('hora',F.hour('eventTimestamp').cast('double'))
    .withColumn('dia_semana',F.dayofweek('eventTimestamp').cast('double'))
    .withColumn('hora_sin',F.sin(F.col('hora')*F.lit(2*math.pi/24)))
    .withColumn('hora_cos',F.cos(F.col('hora')*F.lit(2*math.pi/24))))

def predecir_lote(lote,batch_id):
    if lote.isEmpty():return
    pred=(lote.withColumn('prediction',F.col('velocity').cast('double')) if seleccion=='Persistencia'
          else recargado.transform(lote))
    salida=(pred.select('icao24','eventTimestamp','velocity','prediction')
        .withColumn('modelo',F.lit(seleccion)).withColumn('batch_id',F.lit(batch_id)))
    salida.show(truncate=False)
    salida.write.mode('append').parquet(f'{ROOT}/artifacts/s10/predicciones/jhonatan')
print('Listo para ejecutar la escucha en Kafka')


Listo para ejecutar la escucha en Kafka


In [8]:
# EJECUTAR SOLO DESPUES DE ENTRENAR. Para detener: stream.stop().
stream=(vivos.writeStream.foreachBatch(predecir_lote).outputMode('append')
    .option('checkpointLocation',f'{ROOT}/artifacts/checkpoints/s10_jhonatan_v1')
    .trigger(processingTime='30 seconds').start())
print('Stream activo:',stream.isActive)


Stream activo: True


+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e8cfda|2026-10-30 20:04:13|117.0   |142.58175878600198|LinearRegression|1       |
|e8cfd2|2026-10-30 20:02:48|143.04  |142.62295966448963|LinearRegression|1       |
|e8060b|2026-10-30 20:03:36|248.88  |266.7451328462633 |LinearRegression|1       |
|e80414|2026-10-30 20:04:12|227.6   |237.07210679468025|LinearRegression|1       |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e8cfda|2026-10-30 20:06:15|131.84  |143.29597427988628|LinearRegression|2       |
|e80336|2026-10-30 20:06:16|108.83  |111.73831114219959|LinearRegression|2       |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80336|2026-10-30 20:07:30|81.91   |88.15419740337873 |LinearRegression|3       |
|e8032f|2026-10-30 20:08:11|248.94  |228.03833247829112|LinearRegression|3       |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80336|2026-10-30 20:09:06|64.88   |73.76573937552038 |LinearRegression|4       |
|e8032f|2026-10-30 20:10:13|232.15  |209.38417367122295|LinearRegression|4       |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e8032f|2026-10-30 20:10:49|229.39  |205.06438490561516|LinearRegression|5       |
+------+-------------------+--------+------------------+----------------+--------+



... (43 outputs left out)

+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80270|2026-10-30 21:42:55|141.5   |165.50400789655288|LinearRegression|49      |
|e80274|2026-10-30 21:42:56|188.49  |176.58991457673562|LinearRegression|49      |
|e8021b|2026-10-30 21:42:55|212.81  |231.53970512829906|LinearRegression|49      |
|e80417|2026-10-30 21:42:55|235.21  |226.34111276662193|LinearRegression|49      |
|0ac333|2026-10-30 21:42:56|221.69  |233.1435600743491 |LinearRegression|49      |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80270|2026-10-30 21:44:50|158.32  |181.71444481645318|LinearRegression|50      |
|e80274|2026-10-30 21:44:59|174.48  |169.81145327508608|LinearRegression|50      |
|e8021b|2026-10-30 21:44:58|212.81  |230.94803816317608|LinearRegression|50      |
|e80417|2026-10-30 21:44:59|225.36  |219.62148061447596|LinearRegression|50      |
|aba800|2026-10-30 21:44:59|78.98   |118.14283798571407|LinearRegression|50      |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80270|2026-10-30 21:46:57|192.57  |211.22240724799641|LinearRegression|51      |
|e80274|2026-10-30 21:46:54|155.09  |156.81407938706403|LinearRegression|51      |
|e8021b|2026-10-30 21:46:56|213.74  |231.1073946220748 |LinearRegression|51      |
|e80417|2026-10-30 21:46:16|225.88  |195.9540120237518 |LinearRegression|51      |
|aba800|2026-10-30 21:46:57|139.89  |154.18682073024246|LinearRegression|51      |
+------+-------------------+--------+------------------+----------------+--------+



+------+-------------------+--------+------------------+----------------+--------+
|icao24|eventTimestamp     |velocity|prediction        |modelo          |batch_id|
+------+-------------------+--------+------------------+----------------+--------+
|e80270|2026-10-30 21:49:01|199.49  |209.16238119201216|LinearRegression|52      |
|e80274|2026-10-30 21:49:01|146.83  |146.96681627172052|LinearRegression|52      |
|e8021b|2026-10-30 21:47:56|213.74  |230.81155157382966|LinearRegression|52      |
|e8047a|2026-10-30 21:49:02|222.26  |203.74334700540797|LinearRegression|52      |
|e8032f|2026-10-30 21:48:56|125.42  |147.96607424226934|LinearRegression|52      |
|aba800|2026-10-30 21:49:01|150.54  |185.07453628591867|LinearRegression|52      |
+------+-------------------+--------+------------------+----------------+--------+



## Discusión de resultados
Registra MAE, RMSE y R², diferencia frente a persistencia, cantidad de pares reales y distribución de intervalos. Las predicciones de velocidad futura pueden resultar peores que una persistencia simple. Esa conclusión es válida si se demuestra con pruebas.